# §42 — **K5dfiACCs**: K5dfiACC with the A2 statistics **fitted on ACCIDENT train** (§39 used Nexar's)

Rule: `core/docs/final/KIM_RUN.md` **§42**, written before any number. Run **after** `k5dfi_acc.ipynb` (§39: `ACCIDENT_window`, its CLIP / V2-S s8 caches, KNN, K5dfiACC and its reads must be on Drive). PRELIMINARY (n = 1, s2024). **Printed, decides nothing; K5f stays the reference and the main model.**

| row | recipe | training data | input statistics | here |
|---|---|---|---|---|
| **K5dfiACC** | K5dfi's (§39) | ACCIDENT_window | Nexar train | read only (control) |
| **K5dfiACCs** | K5dfiACC's, config diff **empty** | ACCIDENT_window | **ACCIDENT train** | train + read |
| K5dfi | §34 | Nexar_window | Nexar train | read only (printed) |

Every read of K5dfiACCs is on a bake with **ACCIDENT's** statistics: the four zero-shot bakes K5dfi / K5dfiACC read are re-expressed with `build_v2_inputs restat` (exact for CRN off, gate R2 below), not re-extracted. L0 rows are not re-run: they read plain CLIP, which no statistic touches.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-4 | setup, deps, helpers, preflight (§39 outputs, bakes, K5dfiACC verified) | no | 3-5 min |
| 5 | tests — **raises if red** | no | 1 min |
| 6 | fit on ACCIDENT train -> `ACCIDENT_s8_acc`; **gate R2** (restat of the Nexar-stats bake == the direct bake); restat TAD / DoTA-CAP-dev / DADA / DAD | no | 10-20 min |
| 7 | train K5dfiACCs (K5dfiACC's steps) + guards (J10, config diff to K5dfiACC = {}) | **yes** | 1 × K5dfiACC |
| 8 | reads: TAD, DoTA-CAP-dev, DADA, DAD, ACCIDENT val | yes | 40-60 min |
| 9 | table -> **paste `REPORTS/k5dfiaccs_report.md` to Claude** | no | 2 min |

**Watch the training log: past 2 s/step, stop and paste it.** A re-run skips what is on Drive. Thesis / Thesis-V2 and `kim_accident/` are only read; everything new goes to `Thesis-Final/cache/v2/A2_S/*_acc` and `Thesis-Final/outputs/kim_accident_stats/`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip (DoTA_s1_ncc, DADA2000_orig, TAD_ncc)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window (defs), A2 bakes
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K5dfi / L0dfi / K5f, ACCIDENT caches; every output here
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '## 42. K5dfiACCs' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §42 -- re-upload kat-vad/'
assert 'SUBCOMMAND_RESTAT' in (REPO / 'core/tools/build_v2_inputs.py').read_text(), 'build_v2_inputs has no restat -- re-upload kat-vad/'
assert 'if not is_abnormal:\n            return NORMAL_CLASS' in (REPO / 'core/data/dataset.py').read_text(), \
    'core/data/dataset.py has no §31 fix -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5       = constants.KIM_MAIN_ARM                       # "K5": names run / eval sub-folders only
K5DFI    = constants.KIM_DVS_IGNORE_ROW
K5A      = constants.KIM_ACCIDENT_ROW                   # control: Nexar statistics
K5S      = constants.KIM_ACCIDENT_STATS_ROW             # new: ACCIDENT statistics
SEED     = constants.KIM_SEED
ENC      = constants.V5_ENCODER
OFF      = constants.V2_OFF
IGN      = constants.DVS_ANCHOR_MODE_IGNORE
ACCNAME  = constants.ACCIDENT_DATASET
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
TAD_NAME = constants.TAD_DATASET
STRIDE_B = constants.V2_E1_STRIDE_BC
DAD_NAME, DAD_STRIDE, DAD_SET = constants.DAD_DATASET, constants.DAD_STRIDE, constants.DAD_OUTPUT_SET
OUT_F    = constants.OUTPUT_ROOT
NEX, ACCS = constants.NEXAR_STATS_BAKE_SUFFIX, constants.ACCIDENT_STATS_BAKE_SUFFIX

V2_CACHE = Path(DRIVE_V2) / 'cache'
WINDOW   = Path(DRIVE_V2) / 'data' / f'{constants.NEXAR_DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW}'  # defs of every Nexar read
A2       = V2_CACHE / 'v2' / 'A2_S'
A2_NEXAR = A2 / 'Nexar_s8'
A2_OUT   = constants.CACHE_ROOT / 'v2' / 'A2_S'                 # Thesis-Final: every new bake
ZS_SRC   = {'dada': A2 / f'{T2NAME}_test_sources_nexar', 'dota': A2 / f'{CAP_DEV}_s3_from_s1_nexar',
            'tad': A2 / f'{TAD_NAME}_s8_nexar', DAD_SET: A2_OUT / f'{DAD_NAME}_s{DAD_STRIDE}_nexar'}
assert all(p.name.endswith(NEX) for p in ZS_SRC.values())
ZS_DST   = {s: A2_OUT / (p.name[: -len(NEX)] + ACCS) for s, p in ZS_SRC.items()}
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
T2       = constants.DATA_ROOT / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
TAD      = constants.DATA_ROOT / TAD_NAME
DAD_DATA = Path(DRIVE_F) / 'data' / f'{DAD_NAME}_s{DAD_STRIDE}'
# §39's outputs (read only)
ACC_ROOT  = OUT_F / constants.KIM_ACCIDENT_OUTPUT_DIRNAME
K5A_RUN   = ACC_ROOT / K5A / K5 / f's{SEED}'
K5A_EVAL  = ACC_ROOT / K5A / 'eval'
K5DFI_ROOT = OUT_F / constants.KIM_DVS_IGNORE_OUTPUT_DIRNAME / K5DFI
BUILD    = ACC_ROOT / 'REPORTS' / constants.ACCIDENT_BUILD_REPORT_FILENAME
CLIP_S8  = constants.CLIP_CACHE_DIR / constants.ACCIDENT_CLIP_S8_DATASET
VIDEO_S8 = constants.VIDEO_CACHE_DIR / ENC / f'{constants.ACCIDENT_VIDEO_S8_DATASET}_squash'   # V2-S rows = CLIP s1[::8]
BAKE_NEX = A2_OUT / constants.ACCIDENT_BAKE_DIRNAME          # §39's: Nexar statistics
BAKE_ACC = A2_OUT / constants.ACCIDENT_SELF_BAKE_DIRNAME     # new: fitted on ACCIDENT train
DATA     = Path(DRIVE_F) / 'data' / f'{ACCNAME}_{constants.ACCIDENT_CONSTRUCTION_WINDOW}'
KNN      = constants.CACHE_ROOT / 'knn' / DATA.name / constants.KNN_CACHE_FILENAME   # CLIP s8 only: no statistic in it
ROOT     = OUT_F / constants.KIM_ACCIDENT_STATS_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/k5dfi_acc_stats')
LOCAL.mkdir(parents=True, exist_ok=True)


def row_paths(row):
    run_dir = ROOT / row / K5 / f's{SEED}'
    return run_dir, run_dir / 'checkpoint_last.pt', ROOT / row / 'eval'


INPUTS = (('K5dfiACC run', K5A_RUN / 'verified.json'), ('K5dfiACC config', K5A_RUN / 'config.yaml'),
          *[(f'K5dfiACC {s} read', K5A_EVAL / s / K5) for s in ('dota', 'dada', 'tad', DAD_SET, 'accident_val')],
          *[(f'K5dfi {s} read', K5DFI_ROOT / 'eval' / s / K5) for s in ('dota', 'dada', 'tad', DAD_SET)],
          ('Nexar_window defs', WINDOW / constants.DEFS_FILENAME), ('Nexar A2 bake', A2_NEXAR / constants.V2_INPUT_MANIFEST_FILENAME),
          *[(f'{s} A2 bake (Nexar statistics)', p / constants.V2_INPUT_MANIFEST_FILENAME) for s, p in ZS_SRC.items()],
          ('ACCIDENT A2 bake (Nexar statistics)', BAKE_NEX / constants.V2_INPUT_MANIFEST_FILENAME),
          ('ACCIDENT_window', DATA / constants.LABELS_TRAIN_FILENAME), ('ACCIDENT build report', BUILD),
          ('ACCIDENT CLIP s8', CLIP_S8), ('ACCIDENT V2-S s8', VIDEO_S8), ('ACCIDENT KNN', KNN),
          ('DoTA s1', DOTA_S1), ('T2 meta', T2 / constants.META_FILENAME), ('T2 census', T2_CENSUS),
          ('DADA annotation', ANNOT), ('DoTA metadata', DOTA / 'metadata_val.json'),
          ('TAD labels', TAD / constants.FRAME_LABELS_TEST_FILENAME), ('DAD labels', DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME))
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing (run k5dfi_acc.ipynb first?) -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | runs / reads -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 7-8 need one'
df -h /content | tail -1

In [ ]:
import numpy as np  # noqa: E402
import torch  # noqa: E402
import yaml  # noqa: E402

from core.config import Config                       # noqa: E402
from core.data.v2_splits import load_split           # noqa: E402
from core.tools.accident_build import load_accident_split, read_metadata, source_of  # noqa: E402
from core.tools.dada_whole_eval import test_sources  # noqa: E402
from core.tools.v5_stage1 import _flat               # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    """Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2)."""
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    names = [f.name for f in src.iterdir() if f.is_file()] if ids is None else \
        [f'{v}.npy' for v in ids] + [m.name for p in ('*.json', '*.npz') for m in src.glob(p)]
    todo = [n for n in names if not (dst / n).exists()]
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(lambda n: shutil.copy2(src / n, dst / n), todo))
    print(f'  staged {src.name}: {len(todo)} files in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10)."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def on_drive(folder, min_bytes=1024):
    """Ids of every finished ``.npy`` in ``folder`` from ONE listing (save_array = tmp + rename)."""
    if not folder.is_dir():
        return set()
    with os.scandir(folder) as entries:
        return {e.name[:-4] for e in entries if e.name.endswith('.npy') and e.stat().st_size >= min_bytes}


def metrics_rows(path):
    return [json.loads(l) for l in path.read_text().splitlines() if l.strip()] if path.exists() else []


def ckpt_step(path):
    """The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step)."""
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(run_dir, steps):
    rec, ckpt = run_dir / 'verified.json', run_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def manifest(d):
    return json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())


DEFAULTS = _flat(Config().to_dict())


def config_diff(base, got):
    """Keys that differ; a key added after ``base`` trained may sit at its default."""
    out = {}
    for k in set(base) | set(got):
        if base.get(k) == got.get(k):
            continue
        if k not in base and got.get(k) == DEFAULTS.get(k):
            continue
        if k not in got and base.get(k) == DEFAULTS.get(k):
            continue
        out[k] = (base.get(k), got.get(k))
    return out


# Preflight on small files only (C39)
acc_cfg = _flat(yaml.safe_load((K5A_RUN / 'config.yaml').read_text()))
assert acc_cfg['kim.enabled'] and acc_cfg['v2.motion'] == ENC and acc_cfg['v2.crn'] == OFF and acc_cfg['data.dataset'] == ACCNAME, 'K5A_RUN is not K5dfiACC'
assert acc_cfg['loss.dvs_anchor_mode'] == IGN and not acc_cfg['data.is_egocentric'], 'K5A_RUN is not K5dfiACC'
steps = json.loads((K5A_RUN / 'verified.json').read_text())['step']
assert is_verified(K5A_RUN, steps) and metrics_rows(K5A_RUN / 'metrics.jsonl')[-1]['global_step'] == steps, 'K5dfiACC: J10'
SHA_NEX = manifest(A2_NEXAR)['train_ids_sha1']
for name, d in (('ACCIDENT', BAKE_NEX), *ZS_SRC.items()):
    m = manifest(d)
    assert (m['crn'], m['motion'], m['train_ids_sha1']) == (OFF, ENC, SHA_NEX), f'{name} bake is not Nexar-stats A2: {m}'
build = json.loads(BUILD.read_text())
info, PLAN = build['construction'], build['plan']
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']
assert (STEPS, EPOCHS) == (steps, acc_cfg['train.num_epochs']), f'plan {PLAN} != K5dfiACC ({steps} steps, {acc_cfg["train.num_epochs"]} epochs)'
labels = json.loads((DATA / constants.LABELS_TRAIN_FILENAME).read_text())
windows = json.loads((DATA / constants.WINDOWS_FILENAME).read_text())
TRAIN_SRC = sorted({windows[w]['source'] for w in labels})
ACC_IDS = sorted({windows[w]['source'] for w in windows})           # train + val sources with a window
ACC_VAL = load_accident_split(constants.ACCIDENT_SPLIT_VAL)
assert not set(TRAIN_SRC) & set(ACC_VAL), 'a val clip is in the fit set'
assert set(ACC_IDS) <= on_drive(CLIP_S8) & on_drive(VIDEO_S8) & on_drive(BAKE_NEX), 'ACCIDENT s8 caches incomplete'
DADA_IDS = test_sources(T2)
CAP_IDS = sorted(load_split(CAP_DEV))
ZS_IDS = {'dada': DADA_IDS, 'dota': CAP_IDS,
          'tad': sorted(json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text())),
          DAD_SET: sorted(json.loads((DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text()))}
assert len(DADA_IDS) == 370 and len(ZS_IDS['tad']) == constants.V2_TADZ_VIDEOS and len(ZS_IDS[DAD_SET]) == constants.DAD_TEST_VIDEOS
print(f'preflight OK: K5dfiACC verified ({STEPS} steps) | fit set {len(TRAIN_SRC)} ACCIDENT train clips, bake {len(ACC_IDS)} | '
      f'Nexar statistics {SHA_NEX[:12]}')

## 5. Tests on the touched code (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_v2_restat.py',
     'core/tests/test_accident.py', 'core/tests/test_kim.py'])

## 6. Fit the A2 statistics on ACCIDENT train -> `ACCIDENT_s8_acc`; **gate R2**; restat the four zero-shot bakes

In [ ]:
from core.data.v2_inputs import load_stats  # noqa: E402
from core.data.v2_splits import lines_sha1    # noqa: E402

ids_file = write_ids(LOCAL / 'ids' / 'acc_bake.txt', ACC_IDS)
fit_file = write_ids(LOCAL / 'ids' / 'acc_fit.txt', TRAIN_SRC)
SHA_ACC = lines_sha1(TRAIN_SRC)
L_S8 = stage(CLIP_S8, LOCAL / 'stage' / 'acc_s8', ACC_IDS)
if not ((BAKE_ACC / constants.V2_INPUT_MANIFEST_FILENAME).exists() and set(ACC_IDS) <= on_drive(BAKE_ACC)):
    L_U = stage(VIDEO_S8, LOCAL / 'stage' / 'acc_u', ACC_IDS)
    local = LOCAL / 'bake_acc'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'fit-ids', '--train-ids-file', fit_file, '--ids-file', ids_file,
         '--clip-dir', L_S8, '--video-dir', L_U, '--stride', 1, '--crn', OFF, '--motion', ENC, '--out-dir', local],
        log=LOCAL / 'bake_acc.log')
    publish(local, BAKE_ACC)
    flush_drive()
m = manifest(BAKE_ACC)
assert (m['crn'], m['motion'], m['train_ids_sha1']) == (OFF, ENC, SHA_ACC) and m['baked_ids'] == len(ACC_IDS), f'ACCIDENT bake: {m}'
L_BAKE = stage(BAKE_ACC, LOCAL / 'stage' / 'acc_bake', ACC_IDS)
L_STATS = stage(BAKE_ACC, LOCAL / 'stage' / 'acc_stats', [])          # manifest + statistics only

# Gate R2: the Nexar-stats ACCIDENT bake re-expressed with ACCIDENT's statistics == the direct bake
L_NEX = stage(BAKE_NEX, LOCAL / 'stage' / 'acc_nex', ACC_IDS)
local = LOCAL / 'restat_gate'
shutil.rmtree(local, ignore_errors=True)
run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'restat', '--from-dir', L_NEX, '--stats-dir', L_STATS,
     '--ids-file', ids_file, '--out-dir', local])
old, new = load_stats(L_NEX), load_stats(L_STATS)
D_V = old.m_u.shape[0]                                                # CLIP columns = the first width - D_V
worst, clip_bad = 0.0, []
for v in ACC_IDS:
    a, b, n = np.load(local / f'{v}.npy'), np.load(L_BAKE / f'{v}.npy'), np.load(L_NEX / f'{v}.npy')
    assert a.shape == b.shape == n.shape, (v, a.shape, b.shape, n.shape)
    worst = max(worst, float(np.max(np.abs(a.astype(np.float64) - b))))
    if not np.array_equal(b[:, :-D_V], n[:, :-D_V]):
        clip_bad.append(v)
assert not clip_bad, f'CLIP columns differ between the two ACCIDENT bakes on {clip_bad[:5]}'
assert worst < constants.V2_RESTAT_GATE_ATOL, f'gate R2 FAILED: restat vs direct bake max |diff| {worst:.2e} -- paste to Claude'
print(f'gate R2 OK: restat == direct bake on {len(ACC_IDS)} ACCIDENT clips (max |diff| {worst:.2e}), CLIP columns identical')

ratio = new.sigma_u / old.sigma_u
STATS_LINE = (f'c {old.c:.4f} (Nexar) -> {new.c:.4f} (ACCIDENT); sigma_u ACCIDENT / Nexar median {np.median(ratio):.3f} '
              f'[p10 {np.percentile(ratio, 10):.3f}, p90 {np.percentile(ratio, 90):.3f}]; '
              f'|m_u diff| / sigma_u(Nexar) median {np.median(np.abs(new.m_u - old.m_u) / old.sigma_u):.3f}')
print(STATS_LINE)

# The four zero-shot bakes, re-expressed with ACCIDENT's statistics (exact for CRN off)
for s, src_dir in ZS_SRC.items():
    dst, ids = ZS_DST[s], ZS_IDS[s]
    if not ((dst / constants.V2_INPUT_MANIFEST_FILENAME).exists() and set(ids) <= on_drive(dst)):
        l_src = stage(src_dir, LOCAL / 'stage' / f'zs_{s}_nex', ids)
        local = LOCAL / f'restat_{s}'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'restat', '--from-dir', l_src, '--stats-dir', L_STATS,
             '--ids-file', write_ids(LOCAL / 'ids' / f'zs_{s}.txt', ids), '--out-dir', local], log=LOCAL / f'restat_{s}.log')
        publish(local, dst)
        flush_drive()
    mm, ms = manifest(dst), manifest(src_dir)
    assert (mm['crn'], mm['motion'], mm['train_ids_sha1'], mm['baked_ids']) == (OFF, ENC, SHA_ACC, len(ids)), f'{s}: {mm}'
    assert mm['stride_over_clip_dir'] == ms.get('stride_over_clip_dir'), f'{s}: stride lost in restat'
    print(f'{s}: {len(ids)} items -> {dst.name} (ACCIDENT statistics {SHA_ACC[:12]})')
L_DATA = stage(DATA, LOCAL / 'stage' / 'data')
L_KNN = stage(KNN, LOCAL / 'stage' / 'knn' / KNN.name)

## 7. Train **K5dfiACCs** (§42 R1)

K5dfiACC's `--set` list, unchanged; only `--clip-dir` points at the ACCIDENT-statistics bake. Guards: J10 at the plan's steps, config diff to K5dfiACC **empty**, last-epoch `mul_mil` > 0.01. **Watch the log: past 2 s/step, stop and paste it.**

In [ ]:
K5_BASE = ['train.stage=2', f'train.seed={SEED}', 'train.amp=true',
           f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}', 'kip.enabled=false', f'loss.dvs_anchor_mode={IGN}',
           'data.is_egocentric=false', f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}',
           'dvs.theta=0.7', 'dvs.theta_ego=0.85']
SETS = [*K5_BASE, f'v2.motion={ENC}', f'model.motion_dim={constants.V5_GRID_DIM}', 'kim.enabled=true',
        'kim.prototypes=true', 'kim.edge_mask=true', 'kim.seam_mask=true',
        f'data.dataset={ACCNAME}', f'train.num_epochs={EPOCHS}']             # = K5dfiACC's command (§39 A1)


def check_run(row):
    run_dir, ckpt, _ = row_paths(row)
    mrows = metrics_rows(run_dir / 'metrics.jsonl')
    assert ckpt_step(ckpt) == mrows[-1]['global_step'] == STEPS, f'{row}: J10'
    got = _flat(yaml.safe_load((run_dir / 'config.yaml').read_text()))
    diff = config_diff(acc_cfg, got)
    assert not diff, f'{row} vs {K5A}: config differs on {diff} (§42 R1: only the input cache may differ)'
    last = [r for r in mrows if r['epoch'] == mrows[-1]['epoch']]
    mul_last = sum(r['mul_mil'] for r in last) / len(last)
    mil_last = sum(r['mil'] for r in last) / len(last)
    assert mul_last > constants.KIM_FIXED_MUL_MIL_FLOOR, f'{row}: last-epoch mul_mil {mul_last:.4f}'
    ref = metrics_rows(K5A_RUN / 'metrics.jsonl')
    ref_last = [r for r in ref if r['epoch'] == ref[-1]['epoch']]
    print(f'{row}: config diff to {K5A} {{}} | last-epoch mil {mil_last:.4f} ({K5A} '
          f"{sum(r['mil'] for r in ref_last) / len(ref_last):.4f}) | mul_mil {mul_last:.4f} > {constants.KIM_FIXED_MUL_MIL_FLOOR}")


def train_row(row):
    run_dir, ckpt, _ = row_paths(row)
    if is_verified(run_dir, STEPS):
        print(f'{row} s{SEED}: verified on Drive -- not retrained')
        check_run(row)
        return
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local = LOCAL / 'runs' / row
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in SETS for x in ('--set', s)], '--data-dir', L_DATA,
         '--clip-dir', L_BAKE, '--output-dir', local, '--knn-cache', L_KNN, *resume], log=LOCAL / f'train_{row}.log')
    last = metrics_rows(local / 'metrics.jsonl')[-1]
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{row}: J10 locally {last["global_step"]}'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, run_dir / name)
    flush_drive()
    assert ckpt_step(ckpt) == STEPS, f'{row}: Drive copy is not step {STEPS}'
    (local / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': ckpt.stat().st_size}))
    publish(local / 'verified.json', run_dir / 'verified.json')
    print(f'{row}: trained in {(time.time() - t0) / 60:.0f} min')
    check_run(row)


train_row(K5S)

## 8. Reads on the ACCIDENT-statistics bakes (K5dfi's protocols)

In [ ]:
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
L_TAD = LOCAL / 'TAD_meta'
L_TAD.mkdir(parents=True, exist_ok=True)
for f in TAD.iterdir():                     # json / txt only -- never a frame folder
    if f.is_file() and f.suffix in ('.json', '.txt'):
        shutil.copy2(f, L_TAD / f.name)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in WINDOW.glob('*.json'):             # the definitions every Nexar row read with
    shutil.copy2(f, L_DEFS / f.name)
L_LAB = stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels')
L_DOTA_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS)


def read_evaluate(row, name, dataset, data_dir, clip_dir, n_items, out, ckpt, raw=True):
    if not (out / 'results.json').exists():
        local = LOCAL / 'eval' / row / name
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.evaluate', '--ckpt', ckpt, '--set', f'data.dataset={dataset}',
             '--data-dir', data_dir, '--clip-dir', clip_dir, '--score-norm', 'auto', '--save-scores',
             '--output-dir', local], log=LOCAL / f'{row}_{name}.log')
        publish(local, out)
    r = json.loads((out / 'results.json').read_text())
    assert r['num_videos'] == n_items and (not raw or r['score_norm'] == 'none'), r
    print(f"{row} {name}: macro {r['auc_macro']:.4f} over {r['auc_macro_videos']} | micro {r['auc']:.4f} | AP {r['ap']:.4f}")
    return r


def read_dota(row, ckpt, out, input_dir, plain):
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / row / 'dota'
        shutil.rmtree(local, ignore_errors=True)
        stride = ['--stride', STRIDE_B] if plain else []
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', ckpt, '--input-dir', input_dir,
             '--s1-dir', L_DOTA_S1, '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
             '--data-dir', L_LAB, '--split', CAP_DEV, *stride, '--out-dir', local, '--device', 'auto'], log=LOCAL / f'{row}_dota.log')
        publish(local, out)
    print(f"{row} DoTA-CAP-dev: macro {json.loads((out / 'protocol_b_readout.json').read_text())['macro']['mean']:.4f}")


def read_dada(row, ckpt, out, input_dir):
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / row / 'dada'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', ckpt, '--t2-dir', L_T2,
             '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', input_dir,
             '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / f'{row}_dada.log')
        publish(local, out)
    print(f"{row} DADA whole: macro {json.loads((out / 'dada_readout.json').read_text())['macro']['mean']:.4f}")


VAL = {}
_, ckpt, ev = row_paths(K5S)
IN = {s: stage(ZS_DST[s], LOCAL / 'stage' / f'zs_{s}_acc', ZS_IDS[s]) for s in ZS_DST}
read_evaluate(K5S, 'tad', TAD_NAME, L_TAD, IN['tad'], len(ZS_IDS['tad']), ev / 'tad' / K5 / f's{SEED}', ckpt)
read_dota(K5S, ckpt, ev / 'dota' / K5, IN['dota'], plain=False)
read_dada(K5S, ckpt, ev / 'dada' / K5, IN['dada'])
read_evaluate(K5S, 'dad', DAD_NAME, stage(DAD_DATA, LOCAL / 'stage' / 'dad_meta'), IN[DAD_SET], len(ZS_IDS[DAD_SET]),
              ev / DAD_SET / K5 / f's{SEED}', ckpt)
VAL[K5S] = read_evaluate(K5S, 'accident_val', ACCNAME, L_DATA, L_BAKE, info['val_windows'],
                         ev / 'accident_val' / K5 / f's{SEED}', ckpt, raw=False)
VAL[K5A] = json.loads((K5A_EVAL / 'accident_val' / K5 / f's{SEED}' / 'results.json').read_text())
flush_drive()

## 9. Table -> **paste `REPORTS/k5dfiaccs_report.md` to Claude**

`zs_arm_table` control **K5dfiACC** (rows K5dfiACCs, K5dfi; four sets, macro + micro). Printed only (§42 N).

In [ ]:
local = LOCAL / 'report'
local.mkdir(parents=True, exist_ok=True)
EV = {K5A: K5A_EVAL, K5S: ROOT / K5S / 'eval', K5DFI: K5DFI_ROOT / 'eval'}
reads = [a for row in (K5A, K5S, K5DFI) for s in ('dota', 'dada', 'tad', DAD_SET) for a in ('--read', s, row, EV[row] / s / K5)]
NAME = constants.KIM_ACCIDENT_STATS_TABLE_NAME
run([sys.executable, '-m', 'core.tools.zs_arm_table', '--control', K5A, '--seed', SEED, *reads, '--name', NAME, '--out-dir', local])
lines = ['# K5dfiACCs (KIM_RUN.md §42), s2024, n = 1 — printed, decides nothing', '',
         f'K5dfiACCs = K5dfiACC with the A2 statistics fitted on {len(TRAIN_SRC)} ACCIDENT train clips ({SHA_ACC[:12]}); '
         f'K5dfiACC used Nexar train ({SHA_NEX[:12]}). Config diff = {{}}; {STEPS} steps. Gate R2 max |diff| {worst:.2e}.', '',
         f'Statistics: {STATS_LINE}', '',
         '## ACCIDENT val (in-domain, printed only; each row on its own bake)', '',
         '| row | macro | micro | AP | score norm |', '|---|---|---|---|---|']
lines += [f"| {r} | {v['auc_macro']:.4f} | {v['auc']:.4f} | {v['ap']:.4f} | {v['score_norm']} |" for r, v in VAL.items()]
lines += ['', '## Control K5dfiACC (rows K5dfiACCs, K5dfi)', '', (local / f'{constants.ZS_ARM_TABLE_STEM}_{NAME}.md').read_text()]
(local / f'{constants.KIM_ACCIDENT_STATS_REPORT_STEM}.md').write_text('\n'.join(lines) + '\n')
publish(local, REPORTS)
print((REPORTS / f'{constants.KIM_ACCIDENT_STATS_REPORT_STEM}.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print(f'Drive flushed -- paste REPORTS/{constants.KIM_ACCIDENT_STATS_REPORT_STEM}.md to Claude')